In [2]:
import sys
sys.path.append('../src')

import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

import matplotlib.pyplot as plt
plt.style.core.USER_LIBRARY_PATHS.append('/gscratch/matsulab/utils/ctleelab-mpl-utilities/ctleelab_plothelper')
plt.style.reload_library()

sys.path.append('../../../utils')
from fonts.mpl_fonts import set_arial_font, set_encode_sans_font
# set_arial_font()
set_encode_sans_font()

[mpl_fonts] Using font: Encode Sans Normal
[mpl_fonts] Font dir: /mmfs1/gscratch/matsulab/utils/fonts


In [3]:
## To reload

import importlib
import pinn.plot as plot
import data_generation.preprocess as preprocess
import pinn.utils as utils
import pinn.analysis as analysis
import fonts.mpl_fonts as mpl_fonts

importlib.reload(plot)
importlib.reload(utils)
importlib.reload(preprocess)
importlib.reload(analysis)
importlib.reload(mpl_fonts)

<module 'fonts.mpl_fonts' from '/mmfs1/gscratch/matsulab/sim/pinn-exploration-model/notebooks/../../../utils/fonts/mpl_fonts.py'>

In [15]:
## MAKE RAW MRC MOVIE (FUNCTIONS)

import os
import numpy as np
import mrcfile
import matplotlib.pyplot as plt
import subprocess

def load_mrc(path):
    with mrcfile.open(path, permissive=True) as m:
        vol = m.data.copy()
    return vol

def save_z_frames(vol, out_dir="frames_z", vmin=None, vmax=None, dpi=150):
    set_encode_sans_font()
    os.makedirs(out_dir, exist_ok=True)

    # If data is (z,y,x) this is correct. If not, adjust after printing shape.
    print("volume shape:", vol.shape)

    if vmin is None: vmin = np.percentile(vol, 1)
    if vmax is None: vmax = np.percentile(vol, 99)

    nz = vol.shape[0]
    for z in range(nz):
        img = vol[z, :, :]  # z-slice

        fig = plt.figure(figsize=(4, 4))
        ax = fig.add_subplot(111)
        ax.imshow(img, vmin=vmin, vmax=vmax, cmap="gray_r", alpha=0.8)
        ax.set_axis_off()
        
        # ---- bottom annotation ----
        fig.text(
            0.5, 0.02,                    # (x, y) in figure coords
            f"Slice: {z} / {nz}",     # text
            ha="center",
            va="bottom",
            fontsize=24,
            color="black",
        )

        fn = os.path.join(out_dir, f"frame_{z:04d}.png")
        plt.savefig(fn, dpi=dpi, bbox_inches="tight", pad_inches=0)
        plt.close(fig)

import imageio.v2 as imageio
import os

def frames_to_mp4_imageio(frame_dir, out_mp4, fps=24):
    files = sorted(
        f for f in os.listdir(frame_dir) if f.endswith(".png")
    )
    writer = imageio.get_writer(out_mp4, fps=fps)

    for f in files:
        img = imageio.imread(os.path.join(frame_dir, f))
        writer.append_data(img)

    writer.close()

frames_to_mp4_imageio(frame_dir, f"{output_path}/movie_z.mp4", fps=24)


/gscratch/matsulab/envs/pinn-env/lib/python3.12/subprocess.py:1885: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _fork_exec(


In [36]:
import os
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio

def _slice_indices_pingpong(start_z, end_z, hold_end=0, hold_start=0):
    """
    Sequence:
      start ... end, (hold end), end-1 ... start+1, (hold start)
    Endpoint repeats are controlled by hold_*.
    """
    forward = list(range(start_z, end_z + 1))
    if len(forward) <= 1:
        return forward + [end_z] * max(0, hold_end)

    # reverse without repeating endpoints
    backward = list(range(end_z - 1, start_z, -1))

    seq = forward
    if hold_end > 0:
        seq += [end_z] * hold_end
    seq += backward
    if hold_start > 0:
        seq += [start_z] * hold_start
    return seq


def save_frames_pingpong(
    vol,
    out_dir="frames",
    axis="z",        
    start=0,
    end=None,
    hold_end=0,
    hold_start=0,
    vmin=None,
    vmax=None,
    dpi=150,
    one_based_label=True,
):
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    os.makedirs(out_dir, exist_ok=True)
    set_encode_sans_font()

    # ---- axis handling ----
    axis_map = {"z": 0, "y": 1, "x": 2}
    if axis not in axis_map:
        raise ValueError("axis must be one of 'z', 'y', 'x'")
    ax_idx = axis_map[axis]

    n_slices = vol.shape[ax_idx]

    if end is None:
        end = n_slices - 1

    # clamp + validate
    start = max(0, min(int(start), n_slices - 1))
    end   = max(0, min(int(end),   n_slices - 1))
    if end < start:
        raise ValueError(f"end ({end}) must be >= start ({start})")

    hold_end = max(0, int(hold_end))
    hold_start = max(0, int(hold_start))

    # if vmin is None: vmin = np.percentile(vol, 1)
    # if vmax is None: vmax = np.percentile(vol, 99)

    idx = _slice_indices_pingpong(start, end, hold_end=hold_end, hold_start=hold_start)

    for i, s in enumerate(idx):
        # ---- generic slicing ----
        img = np.take(vol, s, axis=ax_idx)

        fig = plt.figure(figsize=(4, 4))
        ax = fig.add_subplot(111)
        ax.imshow(img, vmin=vmin, vmax=vmax, cmap="gray_r", alpha=1.0)
        ax.set_axis_off()

        label_s = (s + 1) if one_based_label else s
        fig.text(
            0.5, 0.02,
            f"Slice: {label_s} / {n_slices}",
            ha="center", va="bottom",
            fontsize=24, color="black",
        )

        fn = os.path.join(out_dir, f"frame_{i:05d}.png")
        plt.savefig(fn, dpi=dpi, bbox_inches="tight", pad_inches=0)
        plt.close(fig)

    return idx


In [37]:
from pinn.cryoet_io import load_mrc_data

### PARAMETERS ###
GRID_SIZE = 64
axis = "x"
# shape = "czii_gl_1"
shape = "biconcave"
additive = 0.15
missing  = 0.6
str_add  = str(additive).replace('.', '')
str_miss = str(missing).replace('.', '')

# mrc_path = f"../data/experimental/downsampled/{shape}.mrc"
mrc_path = f"../data/synthetic/combine/{shape}_a{str_add}_m{str_miss}.mrc"
vol = load_mrc_data(mrc_path, grid_size=GRID_SIZE)
vol = np.asarray(vol)

# output_path = f"../data/experimental/mrc_video/{shape}"
output_path = f"../data/synthetic/mrc_video/{shape}"
frame_dir = f"{output_path}/frames"
# save_z_frames_pingpong(vol, out_dir=frame_dir, start_z=33, end_z=94, hold_end=24) # for czi_gl_1
save_frames_pingpong(vol, axis = axis, out_dir=frame_dir, start=0, end=64, hold_end=24) 
frames_to_mp4_imageio(frame_dir, f"{output_path}/mrc_{shape}.mp4", fps=24)
print(f"done -> {output_path}/mrc_{shape}.mp4")


[mpl_fonts] Using font: Encode Sans Normal
[mpl_fonts] Font dir: /mmfs1/gscratch/matsulab/utils/fonts


done -> ../data/synthetic/mrc_video/biconcave/mrc_biconcave.mp4
